In [4]:
import pandas as pd
import numpy as np
import re

In [5]:
print("1. Loading Datasets...")
# Update these paths if your files are in Google Drive (e.g., '/content/drive/MyDrive/...')
df_swiggy = pd.read_csv("swiggy_file1.csv")
df_cities = pd.read_csv("Indian Cities Database.csv")

print(f"Initial Swiggy rows: {df_swiggy.shape[0]}")

1. Loading Datasets...
Initial Swiggy rows: 140657


In [6]:
# ==========================================
# PHASE 1: CLEANING SWIGGY DATA
# ==========================================
print("2. Cleaning Swiggy Data...")

# Remove duplicates
df_swiggy.drop_duplicates(inplace=True)

2. Cleaning Swiggy Data...


In [7]:
# Clean 'Average Price' (Convert '₹250 for two' to 250.0)
def clean_price(x):
    if pd.isna(x): return np.nan
    x_str = str(x).replace(',', '')
    match = re.search(r'\d+', x_str)
    return float(match.group()) if match else np.nan

df_swiggy['cost_for_two'] = df_swiggy['Average Price'].apply(clean_price)

In [8]:
# Clean 'Rating' (Convert '--' or 'NEW' to NaN, keep valid numbers)
df_swiggy['rating_clean'] = pd.to_numeric(df_swiggy['Rating'], errors='coerce')

In [9]:
# Clean 'Number of Ratings' (Convert '1K+ ratings' to 1000)
def clean_votes(x):
    if pd.isna(x): return np.nan
    x_str = str(x).lower().replace('ratings', '').replace('+', '').strip()
    if 'k' in x_str:
        try: return float(x_str.replace('k', '')) * 1000
        except: return np.nan
    if x_str == 'too few': return np.nan
    try: return float(x_str)
    except: return np.nan

df_swiggy['votes_clean'] = df_swiggy['Number of Ratings'].apply(clean_votes)

In [10]:
# Drop original messy columns to save memory
df_swiggy.drop(columns=['Average Price', 'Rating', 'Number of Ratings'], inplace=True)

In [11]:
# ==========================================
# PHASE 2: DATA ENRICHMENT (MERGING)
# ==========================================
print("3. Merging with Geographic Data...")

3. Merging with Geographic Data...


In [12]:
# Standardize city names in both datasets for a perfect match (lowercase and strip spaces)
df_swiggy['merge_city'] = df_swiggy['Location'].astype(str).str.lower().str.strip()
df_cities['merge_city'] = df_cities['City'].astype(str).str.lower().str.strip()

In [13]:
# Keep only necessary columns from the cities database
df_cities_subset = df_cities[['merge_city', 'Lat', 'Long', 'State']]

In [14]:
# Perform a Left Join (Keep all Swiggy restaurants, add geo-data where city matches)
df_master = pd.merge(df_swiggy, df_cities_subset, on='merge_city', how='left')

In [15]:
# Drop the temporary merge key
df_master.drop(columns=['merge_city'], inplace=True)

In [16]:
print(f"Final Master Dataset Shape: {df_master.shape}")

Final Master Dataset Shape: (140231, 13)


In [17]:
# Save the final enriched dataset
output_file = "Swiggy_All_India_Enriched.csv"
df_master.to_csv(output_file, index=False)
print(f"✅ Success! Master dataset saved as '{output_file}'")

✅ Success! Master dataset saved as 'Swiggy_All_India_Enriched.csv'
